# Unit 8 solutions: measure the car

These reference solutions fit the longitudinal map and the steering gain from the committed logs, solve the fit again with QR, measure acceleration from positions, run the grip probes, and put the measurements to work.
Every race is seeded, so every number below is reproducible, and executable asserts check each answer against an independent derivation.
Every number is a number about racekit's **carla-like car**: a model, not the truth, and not CARLA and measured values are compared with the values the agents assumed.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.runner import run_race
from racekit.sysid_data import load_log
from racekit.tracks import grip_sections, load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_07, UNIT_08 = AGENTS / "unit-07.py", AGENTS / "unit-08.py"
workdir = Path(mkdtemp(prefix="roar-unit-08-sol-"))
DT = 0.05  # s: one control tick, one log or telemetry row
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
WHEELBASE = 2.875  # m, L (Unit 3)
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on the max |cross-track error|
ASSUMED = (6.0, 8.0, 4.0e-4)  # agents/unit-07.py's A_THROTTLE, A_BRAKE, C_DRAG
MU_G_TEST_CIRCUIT = (7.89, 9.72, 8.65, 7.87, 8.30, 8.14)  # agents/unit-08.py's MU_G (m/s^2)


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # The clean rule: finished, 0 collisions, and max |cross-track error| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


race_log, race_meta = load_log("monza-like-seed0.csv")  # a full noisy race on monza-like
steady_log, steady_meta = load_log("oval-steady-seed0.csv")  # one lap of the oval at a steady 15 m/s

## Exercise 1 - Least squares on six rows

A toy log of six steps of the longitudinal model dv/dt = a_T·throttle − a_B·brake − c·v|v| (no lag):

| step | throttle | brake | mean speed v̄ (m/s) | measured dv/dt (m/s²) |
|---|---|---|---|---|
| 0 | 1.0 | 0.0 | 10 | 5.02 |
| 1 | 0.6 | 0.0 | 20 | 2.85 |
| 2 | 0.0 | 0.5 | 30 | −4.75 |
| 3 | 0.0 | 1.0 | 25 | −9.2275 |
| 4 | 0.3 | 0.0 | 15 | 1.4425 |
| 5 | 1.0 | 0.0 | 5 | 4.9725 |

Row k of the matrix is (throttle_k, −brake_k, −v̄_k·|v̄_k|) and the unknowns are (a_T, a_B, c).

Define:

- `A1`: the 6 × 3 matrix (a NumPy array), and `b1`: the six measured rates;
- `c1`: the least-squares solution, `np.linalg.lstsq(A1, b1, rcond=None)[0]`;
- `residuals1`: A1·c1 − b1 (six numbers), and `rms1`: the root of their mean square (m/s²);
- `sse1`: the sum of squared residuals at `c1`;
- `perpendicular`: the largest |entry| of A1ᵀ·`residuals1` (the residual is perpendicular to every column of A1 at the least-squares solution, so this should be tiny);
- `nudged_sse`: a list of six sums of squared residuals, one for each way of changing **one** coefficient of `c1` by +1% or −1% (a_T up, a_T down, a_B up, a_B down, c up, c down, in that order);
- `always_worse`: `True` when every entry of `nudged_sse` is larger than `sse1`;
- `why_c_is_hard`: one or two sentences: the drag c is the least certain of the three numbers here (compare `c1[2]` with the value the six rows were built from, 3.0e-4, if you like); which column of A1 is to blame, and what would a better log need?

**Prediction.** The rows were built from (a_T, a_B, c) = (5.0, 9.0, 3.0e-4) plus small round-number errors (b1 − A1·(5, 9, 3e-4) shows them), so the fit should land near those, with the drag the least certain: v̄|v̄| is large, but drag itself is a small part of every rate here.

In [ ]:
throttle1 = [1.0, 0.6, 0.0, 0.0, 0.3, 1.0]
brake1 = [0.0, 0.0, 0.5, 1.0, 0.0, 0.0]
speed1 = [10.0, 20.0, 30.0, 25.0, 15.0, 5.0]
A1 = np.array([[t, -b, -v * abs(v)] for t, b, v in zip(throttle1, brake1, speed1)])
b1 = np.array([5.02, 2.85, -4.75, -9.2275, 1.4425, 4.9725])
c1 = np.linalg.lstsq(A1, b1, rcond=None)[0]
residuals1 = A1 @ c1 - b1
rms1 = float(np.sqrt(np.mean(residuals1 ** 2)))
sse1 = float(np.sum(residuals1 ** 2))
perpendicular = float(np.max(np.abs(A1.T @ residuals1)))
nudged_sse = []
for i in range(3):  # a_T, a_B, c
    for factor in (1.01, 0.99):  # up, then down
        nudged = np.array(c1)
        nudged[i] = c1[i] * factor
        nudged_sse.append(float(np.sum((A1 @ nudged - b1) ** 2)))
always_worse = min(nudged_sse) > sse1
why_c_is_hard = (
    "The drag column -v|v| is the culprit: at these speeds (5 to 30 m/s) drag is at most c*900, about 0.27 m/s^2, "
    "and under 0.1 m/s^2 in half the rows, against rate errors of a few hundredths, so the six rows barely see it: "
    "a 1% nudge of c raises the sum of squares hundreds of times less than a 1% nudge of a_T, and c1 lands 9% from "
    "3.0e-4 while a_T and a_B are within 0.6%. "
    "A better log needs many rows at high speed, where v^2 is large, and ideally with the throttle and brake both at "
    "0 (a coast), so that drag is a large part of what the rate measures."
)

print("c1 =", c1)
print(f"RMS {rms1:.4f} m/s^2, SSE {sse1:.6f}, largest |A1^T r| {perpendicular:.1e}")
print("nudged SSE:", [round(value, 6) for value in nudged_sse], "| always worse:", always_worse)
print(why_c_is_hard)

# independent checks: the toy rows' generating values, the perpendicular residual, and the exact cost of a nudge
assert abs(c1[0] - 5.0) < 0.01 and abs(c1[1] - 9.0) < 0.06 and abs(c1[2] - 3.0e-4) / 3.0e-4 > 0.05
assert round(c1[2], 7) == 2.741e-4
assert perpendicular < 1e-8 and abs(rms1 - math.sqrt(sse1 / 6.0)) < 1e-12
for i in range(3):
    # at the minimum the slope is 0, so a nudge of size d along column i costs exactly d^2 * ||A1[:, i]||^2
    for j, factor in enumerate((1.01, 0.99)):
        rise = ((factor - 1.0) * c1[i]) ** 2 * float(np.sum(A1[:, i] ** 2))
        assert abs(nudged_sse[2 * i + j] - sse1 - rise) < 1e-9
assert len(nudged_sse) == 6 and always_worse
assert (nudged_sse[0] - sse1) > 100.0 * (nudged_sse[4] - sse1)  # c is the flattest direction

## Exercise 2 - The lag, and a fit checked on rows it never saw

Use the race log, `race_log` (columns `t, x, y, yaw, vx, vy, throttle, steer, brake`); the speed is `np.hypot(vx, vy)`.

Write `longitudinal_rows(log, lag)`: for every k from `lag` up to `len(log) - 2` such that rows k and k + 1 are one tick apart (|t[k+1] − t[k] − 0.05| < 1e-6) **and** rows k − lag and k are `lag` ticks apart (|t[k] − t[k−lag] − lag·0.05| < 1e-6), one equation: the row (throttle[k − lag], −brake[k − lag], −v̄·|v̄|) with v̄ the mean of the speeds at k and k + 1, and the rate (v[k+1] − v[k]) / 0.05.
It returns `(A, b, k)`, with `k` the array of the rows used.

Define:

- `longitudinal_rows`: the function;
- `lag_rms`: a dict from each lag 0, 1, 2, 3, 4, 5 to the residual RMS of its least-squares fit (m/s²);
- `best_lag`: the lag with the smallest RMS;
- `second_half_fit`: the coefficients (a_T, a_B, c) fitted at `best_lag` on the rows with `t[k] >= t[-1] / 2` only (the second half of the race);
- `first_half_rms_fit`: the RMS of `second_half_fit` on the **other** rows (the first half);
- `first_half_rms_assumed`: the RMS of the assumed map `ASSUMED` on the same first-half rows;
- `full_fit`: the coefficients fitted at `best_lag` on all the rows;
- `percent_apart`: a list of the three differences 100·(second_half_fit − full_fit) / full_fit (%);
- `fig`, `ax`: one plot of `lag_rms` against the lag (markers and a line), with axis labels and a title;
- `what_the_lag_is`: one sentence: what does the best lag say about the car, in seconds?

**Prediction.** The lesson's table has the residual smallest at lag 1 (0.433 m/s²) and growing on either side, so lags 4 and 5 should be worse still; a fit on one half should be as good on the other half as on its own.

In [ ]:
def longitudinal_rows(log, lag):
    t = log["t"]
    speed = np.hypot(log["vx"], log["vy"])
    k = np.arange(lag, len(log) - 1)
    one_tick = np.abs(t[k + 1] - t[k] - DT) < 1e-6
    no_gap = np.abs(t[k] - t[k - lag] - lag * DT) < 1e-6
    k = k[one_tick & no_gap]
    mean = (speed[k] + speed[k + 1]) / 2.0
    A = np.column_stack([log["throttle"][k - lag], -log["brake"][k - lag], -mean * np.abs(mean)])
    b = (speed[k + 1] - speed[k]) / DT
    return A, b, k


def rms(A, b, c):
    return float(np.sqrt(np.mean((A @ np.array(c) - b) ** 2)))


lag_rms = {}
for lag in range(6):
    A, b, _ = longitudinal_rows(race_log, lag)
    lag_rms[lag] = rms(A, b, np.linalg.lstsq(A, b, rcond=None)[0])
best_lag = min(lag_rms, key=lag_rms.get)

A, b, k = longitudinal_rows(race_log, best_lag)
second_half = race_log["t"][k] >= race_log["t"][-1] / 2.0
first_half = ~second_half
second_half_fit = np.linalg.lstsq(A[second_half], b[second_half], rcond=None)[0]
first_half_rms_fit = rms(A[first_half], b[first_half], second_half_fit)
first_half_rms_assumed = rms(A[first_half], b[first_half], ASSUMED)
full_fit = np.linalg.lstsq(A, b, rcond=None)[0]
percent_apart = [float(value) for value in 100.0 * (second_half_fit - full_fit) / full_fit]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(list(lag_rms), [lag_rms[lag] for lag in lag_rms], marker="o")
ax.set_xlabel("lag (ticks of 0.05 s)")
ax.set_ylabel("residual RMS (m/s^2)")
ax.set_title("monza-like-seed0.csv: the residual chooses the lag")
plt.show()
what_the_lag_is = (
    "The best lag is 1 tick: the speed change from k to k + 1 follows the command logged one tick earlier, so the "
    "car applies its throttle and brake about 0.05 s after the agent sends them."
)

print({lag: round(value, 3) for lag, value in lag_rms.items()}, "best lag:", best_lag)
print("fit on the second half:", second_half_fit, "| on all rows:", full_fit)
print(f"first half: RMS {first_half_rms_fit:.4f} (the fit) against {first_half_rms_assumed:.4f} (the assumed map)")
print("percent apart:", [round(value, 3) for value in percent_apart])
print(what_the_lag_is)

# independent checks: the lesson's lag table, the fit beats the guess on rows it never saw, the halves agree
assert [round(lag_rms[lag], 3) for lag in range(4)] == [0.570, 0.433, 0.535, 0.596]
assert best_lag == 1 and lag_rms[5] > lag_rms[4] > lag_rms[3]
assert first_half_rms_fit < first_half_rms_assumed - 0.1
assert abs(first_half_rms_fit - lag_rms[1]) < 0.01  # as good on unseen rows as on all of them
assert max(abs(value) for value in percent_apart) < 1.0
assert [round(full_fit[0], 2), round(full_fit[1], 2), round(full_fit[2], 6)] == [5.17, 8.54, 3.08e-4]
row = 1000  # one equation built by hand from the log
speed = np.hypot(race_log["vx"], race_log["vy"])
kk = k[row]
mean = (speed[kk] + speed[kk + 1]) / 2.0
assert A[row][0] == race_log["throttle"][kk - 1] and A[row][2] == -mean * abs(mean)
assert abs(b[row] - (speed[kk + 1] - speed[kk]) / 0.05) < 1e-9

## Exercise 3 - QR, and a log that cannot see the brake

Write `back_substitute(r, y)`: solve r·c = y for an upper-triangular m × m matrix r by back substitution (the last unknown first: c[m−1] = y[m−1] / r[m−1, m−1], then upwards, each row using the unknowns already found), returning a NumPy array.
Check it on r = [[2, 1, −1], [0, 3, 2], [0, 0, 4]] and y = [3, 13, 8] before you trust it.

Then, with Exercise 2's `longitudinal_rows(race_log, 1)` as (A, b):

- `q`, `r` = `np.linalg.qr(A)`;
- `c_qr` = `back_substitute(r, q.T @ b)`;
- `largest_gap`: the largest |`c_qr` − the `lstsq` solution|, relative to the `lstsq` solution;
- `constant` = ‖b‖² − ‖qᵀb‖²;
- `identity_errors`: for each of the five coefficient vectors `c_qr`, `ASSUMED`, (0, 0, 0), (5, 9, 0) and (1, 1, 1), the |difference| between ‖Ac − b‖² and ‖rc − qᵀb‖² + `constant`, divided by ‖Ac − b‖² (a list of five numbers).

Now do the same fit on the **steady** log, `steady_log`, with the same function (lag 1).
That lap holds a steady 15 m/s: the speed loop barely brakes.

- `steady_fit`: its least-squares (a_T, a_B, c);
- `race_r_diagonal`, `steady_r_diagonal`: the three diagonal entries of r for the race log and for the steady log (`[r[i, i] for i in range(3)]`);
- `excitation`: two or three sentences: compare the brake entry of the two diagonals and the two fitted a_B; what does a small diagonal entry of r mean for back substitution, and what does a log need to contain before a coefficient can be identified from it?

**Prediction.** QR and back substitution solve the same problem as `lstsq`, so they agree to rounding, and the identity holds at every c, not just at the answer. The steady lap barely brakes, so its R should have a tiny brake entry.

In [ ]:
def back_substitute(r, y):
    m = len(y)
    c = np.zeros(m)
    for i in range(m - 1, -1, -1):  # the last unknown first
        known = sum(r[i, j] * c[j] for j in range(i + 1, m))
        c[i] = (y[i] - known) / r[i, i]
    return c


r_small = np.array([[2.0, 1.0, -1.0], [0.0, 3.0, 2.0], [0.0, 0.0, 4.0]])
y_small = np.array([3.0, 13.0, 8.0])
small_check = back_substitute(r_small, y_small)
A, b, _ = longitudinal_rows(race_log, 1)
q, r = np.linalg.qr(A)
c_qr = back_substitute(r, q.T @ b)
largest_gap = float(np.max(np.abs(c_qr - full_fit) / np.abs(full_fit)))
constant = float(np.sum(b ** 2) - np.sum((q.T @ b) ** 2))
identity_errors = []
for c in (c_qr, np.array(ASSUMED), np.zeros(3), np.array([5.0, 9.0, 0.0]), np.array([1.0, 1.0, 1.0])):
    left = float(np.sum((A @ c - b) ** 2))
    right = float(np.sum((r @ c - q.T @ b) ** 2)) + constant
    identity_errors.append(abs(left - right) / left)

A_steady, b_steady, _ = longitudinal_rows(steady_log, 1)
steady_fit = np.linalg.lstsq(A_steady, b_steady, rcond=None)[0]
_, r_steady = np.linalg.qr(A_steady)
race_r_diagonal = [float(r[i, i]) for i in range(3)]
steady_r_diagonal = [float(r_steady[i, i]) for i in range(3)]
excitation = (
    "The brake entry of R is 27.7 in size for the race and only 0.77 for the steady lap (about 36 times smaller), "
    "because the steady lap's largest brake is 0.057: its fitted a_B, 8.27, comes from almost no braking at all, "
    "and its drag even comes out negative, because at a steady 15 m/s the v|v| column hardly changes. "
    "Back substitution divides by each diagonal entry, so a small entry turns small errors in Q^T b into a large "
    "error in that coefficient. "
    "A coefficient can be identified only from a log in which its own input varies, strongly and not in step with "
    "the others: hard braking for a_B, a wide range of speeds for the drag."
)

print("back substitution on the 3 x 3 example:", small_check)
print("QR + back substitution:", c_qr, f"| largest relative gap to lstsq {largest_gap:.1e}")
print(f"constant {constant:.2f}; identity errors", [f"{value:.1e}" for value in identity_errors])
print("steady fit:", steady_fit)
print("diagonal of R, race:", [round(value, 3) for value in race_r_diagonal],
      "| steady:", [round(value, 3) for value in steady_r_diagonal])
print(excitation)

# independent checks: the toy system by hand, R is triangular, the identity, and the brake's tiny diagonal
assert np.max(np.abs(small_check - np.array([1.0, 3.0, 2.0]))) < 1e-12
assert np.max(np.abs(r_small @ small_check - y_small)) < 1e-12
assert r[1, 0] == 0.0 and r[2, 0] == 0.0 and r[2, 1] == 0.0
assert largest_gap < 1e-9 and max(identity_errors) < 1e-9 and len(identity_errors) == 5
assert abs(constant - float(np.sum((A @ full_fit - b) ** 2))) < 1e-9 * constant  # the fit's own SSE
assert abs(steady_r_diagonal[1]) * 20.0 < abs(race_r_diagonal[1])
assert np.max(steady_log["brake"]) < 0.06 and steady_fit[2] < 0.0
assert [round(abs(value), 2) for value in (race_r_diagonal[1], steady_r_diagonal[1])] == [27.70, 0.77]

## Exercise 4 - The steering gain, and which rows to trust

Write `steering_gain(log, lag=1, stillness_limit=0.002)` for a sensor log:

- the yaw rate over k → k + 1 is the wrapped heading change, ((yaw[k+1] − yaw[k] + π) mod 2π − π) / 0.05;
- the speed is the mean of `np.hypot(vx, vy)` at k and k + 1;
- the steer paired with k → k + 1 is the one logged at k − lag (0 for the first `lag` transitions);
- `stillness` at k is the standard deviation of that paired steer over k − 10 … k (fewer at the start);
- a row counts when the speed is above 5 m/s, |steer| > 0.005 and `stillness < stillness_limit`;
- the wheel angle of a counted row is atan(r·L / v), with L = `WHEELBASE` (`math.atan2(r * WHEELBASE, v)`), and the gain is the one-column least-squares fit of wheel angle = gain · (−steer).

It returns `(the number of counted rows, the gain in degrees)`, and `(count, None)` when 20 rows or fewer count.

Define:

- `steering_gain`: the function;
- `steady_gain`: a dict from each lag 0, 1, 2 to the gain on `steady_log` (degrees, `stillness_limit` 0.002);
- `race_gain`: a dict from each `stillness_limit` 0.002, 0.01 and 0.05 to the result `(rows, gain)` on `race_log` at lag 1;
- `trusted`: the steady log's gain at lag 1;
- `why_still`: two or three sentences: why does the race log's gain fall as the limit is loosened, and why is the steady log the one to trust? (Think about what the tyres do just after the steer changes, Unit 6.)

Then the size of the error the assumed 70° makes.
The controller wants a wheel angle of 5°, so it sends steer = −5 / 70; the car turns its wheels −steer × `trusted` degrees.

- `wheel_angle_got`: that wheel angle (degrees);
- `radius_wanted`, `radius_got`: the turning radii L / tan(δ) of the two wheel angles (m).

**Prediction.** The lesson measured 65.71° at lag 1 from the steady lap; on the race log the steer is rarely still, and rows taken while it changes should drag the gain down.

In [ ]:
def steering_gain(log, lag=1, stillness_limit=0.002):
    yaw = log["yaw"]
    rate = ((yaw[1:] - yaw[:-1] + math.pi) % (2.0 * math.pi) - math.pi) / DT  # wrapped, over k -> k + 1
    speed = np.hypot(log["vx"], log["vy"])
    mean_speed = (speed[:-1] + speed[1:]) / 2.0
    steer = log["steer"]
    paired = np.concatenate([np.zeros(lag), steer[:len(steer) - 1 - lag]])  # the steer logged at k - lag
    stillness = np.array([np.std(paired[max(0, k - 10):k + 1]) for k in range(len(paired))])
    counts = (mean_speed > 5.0) & (np.abs(paired) > 0.005) & (stillness < stillness_limit)
    rows = int(np.sum(counts))
    if rows <= 20:
        return rows, None
    wheel_angle = np.array([math.atan2(r_k * WHEELBASE, v_k) for r_k, v_k in zip(rate[counts], mean_speed[counts])])
    gain = np.linalg.lstsq(np.column_stack([-paired[counts]]), wheel_angle, rcond=None)[0][0]
    return rows, math.degrees(gain)


steady_gain = {lag: steering_gain(steady_log, lag)[1] for lag in (0, 1, 2)}
race_gain = {limit: steering_gain(race_log, 1, limit) for limit in (0.002, 0.01, 0.05)}
trusted = steady_gain[1]
why_still = (
    "Just after the steer changes, the tyres have not yet built up their force (Unit 6), so the yaw rate lags the "
    "command and the wheel angle measured from it is smaller than the gain times the steer. "
    "Loosening the limit lets in more and more of those changing-steer rows (1311, then 3359, then 4276 of them), "
    "and they pull the race log's gain down from 65.3 to 58.7 to 50.8 degrees. "
    "The steady lap holds an almost constant steer through long bends, so its counted rows are steady turns where "
    "the kinematic relation holds, and its gain barely moves with the lag (65.65 to 65.74 degrees)."
)
wheel_angle_got = 5.0 / 70.0 * trusted
radius_wanted = WHEELBASE / math.tan(math.radians(5.0))
radius_got = WHEELBASE / math.tan(math.radians(wheel_angle_got))

print("steady log, gain by lag:", {lag: round(gain, 3) for lag, gain in steady_gain.items()})
print("race log, lag 1, by stillness limit:", {limit: (rows, round(gain, 2)) for limit, (rows, gain) in race_gain.items()})
print(f"trusted: {trusted:.2f} deg (assumed: 70)")
print(f"wanted 5 deg -> got {wheel_angle_got:.3f} deg; radius wanted {radius_wanted:.2f} m, got {radius_got:.2f} m")
print(why_still)

# independent checks: the lesson's measured gain, the falling race gain, and the small-angle size of the error
assert round(trusted, 2) == 65.71
assert max(steady_gain.values()) - min(steady_gain.values()) < 0.1
gains = [race_gain[limit][1] for limit in (0.002, 0.01, 0.05)]
rows = [race_gain[limit][0] for limit in (0.002, 0.01, 0.05)]
assert gains[0] > gains[1] > gains[2] and rows[0] < rows[1] < rows[2]
assert [round(gain, 1) for gain in gains] == [65.3, 58.7, 50.8]
assert abs(wheel_angle_got - 5.0 * trusted / 70.0) < 1e-12 and round(wheel_angle_got, 2) == 4.69
assert abs(radius_got / radius_wanted - 70.0 / trusted) < 0.005  # tan is nearly linear at 5 degrees
assert radius_got - radius_wanted > 2.0  # every corner is drawn over 2 m wider than the controller asks

## Exercise 5 - Acceleration from positions

**First, check the formula where you know the answer.**
A car drives a circle of radius R = 100 m at v = 20 m/s, and its position is sampled every 0.05 s for 10 s (201 samples): at time t_j = 0.05·j the position is (R cos(v t_j / R), R sin(v t_j / R)).

- `circle_accel`: the |a| of the second difference, (p_{j+1} − 2p_j + p_{j−1}) / 0.05², at every inner sample (a NumPy array of 199 values);
- `circle_exact`: v² / R;
- `circle_error`: the largest |`circle_accel` − `circle_exact`| / `circle_exact`.

**Then write the probe's row rule** as a function, for racekit telemetry (`result.telemetry`):
`section_p99(telemetry, track, min_rows=20)` returns a list of six pairs `(rows, p99)`, one per grip section, where:

- |a| at row i (for i from 1 to N − 2) is the second difference of (x, y) at rows i − 1, i, i + 1;
- row i qualifies when rows i − 1, i and i + 1 all have `held == 0`, neither of the two steps between them is longer than 5 m, and `v` at row i is above 5 m/s;
- row i's section is `grip_sections(track)[(progress[i] + 1) % n]` (Unit 7: racekit's progress p is waypoint p + 1);
- `rows` is the number of qualifying rows in the section, and `p99` the 99th percentile of their |a| (`np.percentile(values, 99)`), or `None` when `rows <= min_rows`.

Race `agents/unit-08.py` on `"test-circuit"`, `seed=0`, `max_seconds=450.0` (a clean race), and define:

- `section_p99`: the function;
- `clean_race`: the `RaceResult`;
- `clean_race_p99`: `section_p99(clean_race.telemetry, load_track("test-circuit"))`;
- `fraction_of_measured`: a list of the six ratios p99 / `MU_G_TEST_CIRCUIT[s]`;
- `any_above`: `True` when any ratio is above 1;
- `lower_bound`: two sentences: what does `clean_race_p99` tell you about the grip, and why can it not be an estimate of it, however many clean races you log?

**Prediction.** The second difference of a circle points to its centre with length (2R/Δt²)(1 − cos(vΔt/R)), within about (vΔt/R)²/12 ≈ 8e-6 of v²/R. A clean race never slides, so its per-section numbers should all sit below the probed `MU_G`.

In [ ]:
R_CIRCLE, V_CIRCLE = 100.0, 20.0
t_circle = DT * np.arange(201)
px = R_CIRCLE * np.cos(V_CIRCLE * t_circle / R_CIRCLE)
py = R_CIRCLE * np.sin(V_CIRCLE * t_circle / R_CIRCLE)
circle_accel = np.hypot(px[2:] - 2.0 * px[1:-1] + px[:-2], py[2:] - 2.0 * py[1:-1] + py[:-2]) / DT ** 2
circle_exact = V_CIRCLE ** 2 / R_CIRCLE
circle_error = float(np.max(np.abs(circle_accel - circle_exact)) / circle_exact)


def section_p99(telemetry, track, min_rows=20):
    x, y, held, v = telemetry["x"], telemetry["y"], telemetry["held"], telemetry["v"]
    accel = np.hypot(x[2:] - 2.0 * x[1:-1] + x[:-2], y[2:] - 2.0 * y[1:-1] + y[:-2]) / DT ** 2  # rows 1 .. N - 2
    step = np.hypot(np.diff(x), np.diff(y))  # step j is from row j to row j + 1
    qualifies = (held[:-2] == 0) & (held[1:-1] == 0) & (held[2:] == 0)
    qualifies = qualifies & (step[:-1] <= 5.0) & (step[1:] <= 5.0) & (v[1:-1] > 5.0)
    sections = np.array(grip_sections(track))
    row_section = sections[(telemetry["progress"][1:-1] + 1) % len(sections)]
    pairs = []
    for s in range(6):
        values = accel[qualifies & (row_section == s)]
        pairs.append((len(values), float(np.percentile(values, 99)) if len(values) > min_rows else None))
    return pairs


test_circuit = load_track("test-circuit")
clean_race = run_race(UNIT_08, "test-circuit", car=carla_like(noise=True), seed=0, max_seconds=450.0)
clean_race_p99 = section_p99(clean_race.telemetry, test_circuit)
fraction_of_measured = [p99 / MU_G_TEST_CIRCUIT[s] for s, (_, p99) in enumerate(clean_race_p99)]
any_above = max(fraction_of_measured) > 1.0
lower_bound = (
    "The clean race never slid, so each section's 99th percentile (between 0.83 and 0.91 of the probed grip) says "
    "only that the tyres gave at least that much: mu*g is at least p99 in every section. "
    "However many clean races you log, the planner asked for f times the grip and the car never went past it, so "
    "the numbers measure the plan, not the tyres' limit; only a race that reaches the limit, and slides, can "
    "estimate it."
)

print(f"circle: |a| from {np.min(circle_accel):.6f} to {np.max(circle_accel):.6f}, exact {circle_exact}, "
      f"error {circle_error:.1e}")
print(f"clean race: clean {clean(clean_race)}, {clean_race.total_time:.2f} s")
print("per section (rows, p99):", [(rows, round(p99, 2)) for rows, p99 in clean_race_p99])
print("fraction of the measured grip:", [round(value, 3) for value in fraction_of_measured], "| any above 1:", any_above)
print(lower_bound)

# independent checks: the circle's exact second difference, the reference race, and a row-by-row recount
omega = V_CIRCLE / R_CIRCLE
assert len(circle_accel) == 199 and circle_exact == 4.0
assert np.max(np.abs(circle_accel - 2.0 * R_CIRCLE * (1.0 - math.cos(omega * DT)) / DT ** 2)) < 1e-9
assert circle_error < 1e-5  # about (omega * DT)^2 / 12
assert clean(clean_race) and abs(clean_race.total_time - 301.95) <= TICK
telemetry = clean_race.telemetry
sections = grip_sections(test_circuit)
counts = [0] * 6
for i in range(1, len(telemetry["t"]) - 1):
    still = telemetry["held"][i - 1] == 0 and telemetry["held"][i] == 0 and telemetry["held"][i + 1] == 0
    short = (math.hypot(telemetry["x"][i] - telemetry["x"][i - 1], telemetry["y"][i] - telemetry["y"][i - 1]) <= 5.0
             and math.hypot(telemetry["x"][i + 1] - telemetry["x"][i], telemetry["y"][i + 1] - telemetry["y"][i]) <= 5.0)
    if still and short and telemetry["v"][i] > 5.0:
        counts[sections[(telemetry["progress"][i] + 1) % len(sections)]] += 1
assert counts == [rows for rows, _ in clean_race_p99] and min(counts) > 20
assert not any_above and min(fraction_of_measured) > 0.8

## Exercise 6 - Two probes of your own

Probe two sections of `test-circuit-seed1`, a track of the training family: sections **2** and **5**.
A probe is a variant of `agents/unit-07.py` with `A_LAT = 3.5` and `GRIP` = 1.0 everywhere except 3.5 in the pushed section, raced on `"test-circuit-seed1"` with `laps=1`, `max_seconds=300.0`, `seed=0`.
Use Exercise 5's `section_p99` and keep only the pushed section's pair.

The lesson's rules: a probe that crashed gives an **estimate**; a probe that never crashed gives a **lower bound**; 20 qualifying rows or fewer give **no estimate**.

Define:

- `probes`: a dict from the section (2 and 5) to its `RaceResult`;
- `my_mu_g`: a dict from the section to its p99 (m/s², rounded to 0.01);
- `kind`: a dict from the section to `"estimate"`, `"lower bound"` or `"no estimate"`;
- `matches_record`: `True` when both values equal the lesson's table for `test-circuit-seed1` (9.36 and 7.56) to 0.01.

Now what the numbers buy.
With κ the Menger curvature of `test-circuit-seed1`'s waypoints (cyclic, as in Unit 2), find the **tightest** waypoint of each of the two sections (the largest |κ| among the waypoints whose `grip_sections` value is that section), and compare the corner cap √(a / |κ|) for three lateral limits a:

- `tightest`: a dict from the section to (that waypoint's index, its |κ| in 1/m);
- `cap_unit_7`: a dict from the section to the cap with Unit 7's a = 4.6 (m/s);
- `cap_unit_8`: a dict from the section to the cap with a = 0.56 · `my_mu_g[section]` (m/s);
- `cap_full`: a dict from the section to the cap with a = `my_mu_g[section]`, all of the measured grip (m/s);
- `which_section_gains`: one or two sentences: in which section does Unit 8's cap rise above Unit 7's, in which does it fall, and why is a fall here not a loss?

**Prediction.** With the lesson's own probe and seed, the two sections should reproduce the table's 9.36 and 7.56 m/s², and both probes should crash.

In [ ]:
def probe_kind(result, rows):
    # The lesson's rules: too few rows -> no estimate; crashed -> an estimate; never crashed -> a lower bound.
    if rows <= 20:
        return "no estimate"
    return "estimate" if result.collisions >= 1 else "lower bound"


seed1 = load_track("test-circuit-seed1")
probes, my_mu_g, kind = {}, {}, {}
for s in (2, 5):
    grip = [1.0] * 6
    grip[s] = 3.5
    probes[s] = run_race(variant(UNIT_07, f"probe-seed1-{s}", A_LAT=3.5, GRIP=tuple(grip)), "test-circuit-seed1",
                         car=carla_like(noise=True), laps=1, max_seconds=300.0, seed=0)
    rows, p99 = section_p99(probes[s].telemetry, seed1)[s]
    my_mu_g[s] = round(p99, 2)
    kind[s] = probe_kind(probes[s], rows)
matches_record = abs(my_mu_g[2] - 9.36) < 0.005 and abs(my_mu_g[5] - 7.56) < 0.005

xs, ys = seed1.x, seed1.y
n = len(xs)
before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
twice_area = (xs - xs[before]) * (ys[after] - ys[before]) - (ys - ys[before]) * (xs[after] - xs[before])
sides = (np.hypot(xs - xs[before], ys - ys[before]) * np.hypot(xs[after] - xs, ys[after] - ys)
         * np.hypot(xs[after] - xs[before], ys[after] - ys[before]))
kappa = np.abs(2.0 * twice_area / sides)  # Unit 2's Menger curvature, cyclic
seed1_sections = np.array(grip_sections(seed1))
tightest, cap_unit_7, cap_unit_8, cap_full = {}, {}, {}, {}
for s in (2, 5):
    inside = np.nonzero(seed1_sections == s)[0]
    index = int(inside[np.argmax(kappa[inside])])
    tightest[s] = (index, float(kappa[index]))
    cap_unit_7[s] = math.sqrt(4.6 / kappa[index])
    cap_unit_8[s] = math.sqrt(0.56 * my_mu_g[s] / kappa[index])
    cap_full[s] = math.sqrt(my_mu_g[s] / kappa[index])
which_section_gains = (
    "Unit 8's cap rises above Unit 7's in section 2 (22.2 against 20.8 m/s), whose grip is high, and falls in "
    "section 5 (16.6 against 17.3 m/s), whose grip is low. "
    "The fall is not a loss: Unit 7's 4.6 m/s^2 is 0.61 of section 5's measured 7.56, past the 0.58 edge of the "
    "family, so the speed it gives up there is margin Unit 7 never really had, and the grippier sections pay it back "
    "(Exercise 7: Unit 8 is the faster of the two on this track)."
)

for s in (2, 5):
    rows, _ = section_p99(probes[s].telemetry, seed1)[s]
    print(f"section {s}: {probes[s].collisions} collisions, {rows} rows, mu*g {my_mu_g[s]} ({kind[s]}); "
          f"tightest waypoint {tightest[s][0]}, |kappa| {tightest[s][1]:.5f} 1/m; "
          f"caps {cap_unit_7[s]:.2f} (Unit 7), {cap_unit_8[s]:.2f} (Unit 8), {cap_full[s]:.2f} (all the grip) m/s")
print("matches the lesson's table:", matches_record)
print(which_section_gains)

# independent checks: both probes slid, the record, and the caps' order
assert kind == {2: "estimate", 5: "estimate"} and matches_record
assert min(probes[s].collisions for s in (2, 5)) >= 1
for s in (2, 5):
    assert seed1_sections[tightest[s][0]] == s
    assert max(kappa[seed1_sections == s]) == tightest[s][1]
    assert cap_unit_8[s] < cap_full[s] and abs(cap_unit_8[s] / cap_full[s] - math.sqrt(0.56)) < 1e-12
assert cap_unit_8[2] > cap_unit_7[2] and cap_unit_8[5] < cap_unit_7[5]
assert [round(cap_unit_7[s], 1) for s in (2, 5)] == [20.8, 17.3]
assert [round(cap_unit_8[s], 1) for s in (2, 5)] == [22.2, 16.6]
assert round(4.6 / my_mu_g[5], 2) == 0.61

## Exercise 7 - Which fix fixed it?

On `test-circuit-seed1`, race seed **1** is one where Unit 7's agent is not clean.
Two things changed between Unit 7 and Unit 8: the steering gain (70° → 66°) and the grip (A_LAT → `MU_FRACTION` · `MU_G`).
Separate them with three races on `"test-circuit-seed1"`, `seed=1`, `max_seconds=450.0`:

- `unit_7`: `agents/unit-07.py` as shipped;
- `unit_7_at_66`: `agents/unit-07.py` with `MAX_STEER = math.radians(66.0)` only;
- `unit_8`: `agents/unit-08.py` with `test-circuit-seed1`'s measured grip, `MU_G = (7.72, 9.80, 9.36, 8.38, 9.53, 7.56)` (the lesson's table), and its shipped f = 0.56.

Define:

- the three `RaceResult` objects, `unit_7`, `unit_7_at_66` and `unit_8`;
- `outcome`: a dict from `"unit-07"`, `"unit-07 at 66"` and `"unit-08"` to a tuple (clean, race time in s, max |cross-track error| in m);
- `steering_alone_enough`: `True` when `unit_7_at_66` is clean;
- `grip_gain_s`: the race time of `unit_7_at_66` minus that of `unit_8` (s), or `None` unless both are clean;
- `attribution`: two or three sentences: on this track and seed, which change made the agent clean and which made it faster? Would you expect the same answer on `test-circuit-seed2026` (the lesson's §5)?

**Prediction.** Run in order of the two changes: if 66° alone makes Unit 7 clean here, the steering was the cause of the crash and the grip is a speed gain on top.

In [ ]:
SEED1_MU_G = (7.72, 9.80, 9.36, 8.38, 9.53, 7.56)  # the lesson's table


def seed1_race(source):
    return run_race(source, "test-circuit-seed1", car=carla_like(noise=True), seed=1, max_seconds=450.0)


unit_7 = seed1_race(UNIT_07)
unit_7_at_66 = seed1_race(variant(UNIT_07, "unit-07-66deg", MAX_STEER=math.radians(66.0)))
unit_8 = seed1_race(variant(UNIT_08, "unit-08-seed1", MU_G=SEED1_MU_G))
outcome = {name: (clean(result), result.total_time, max_offset(result))
           for name, result in (("unit-07", unit_7), ("unit-07 at 66", unit_7_at_66), ("unit-08", unit_8))}
steering_alone_enough = clean(unit_7_at_66)
grip_gain_s = unit_7_at_66.total_time - unit_8.total_time if clean(unit_7_at_66) and clean(unit_8) else None
attribution = (
    "On test-circuit-seed1, seed 1, the steering gain alone made the agent clean: Unit 7 with only 66 degrees "
    "finishes clean in 302.00 s, so the assumed 70 degrees is what made it crash. "
    "The measured grip made it faster, 4.35 s, by using more of the grip where the grip is high. "
    "On test-circuit-seed2026 I would not expect the same answer: the lesson's §5 shows Unit 7 at 66 degrees "
    "still crashes there on every seed, because section 1 there has less grip than Unit 7's fixed 4.6 m/s^2 can "
    "safely use, and only the measured grip makes it clean."
)

for name, (is_clean, time_s, offset) in outcome.items():
    print(f"{name:14s}: clean {is_clean}, {time_s:.2f} s, max |offset| {offset:.2f} m")
print(f"steering alone enough: {steering_alone_enough}; grip gain {grip_gain_s:.2f} s")
print(attribution)

# independent checks: the lesson's grid gives Unit 8 at f 0.56 on seed1 a median of 297.65 s
assert not outcome["unit-07"][0] and unit_7.collisions >= 1
assert steering_alone_enough and abs(unit_7_at_66.total_time - 302.00) <= TICK
assert clean(unit_8) and abs(unit_8.total_time - 297.65) <= TICK
assert abs(grip_gain_s - 4.35) <= 2.0 * TICK

## Exercise 8 - Challenge: the drag from a coast-down alone

A classic way to measure drag: get the car moving, then **coast**, no throttle and no brake, and watch it slow down.
With throttle = brake = 0 the model is dv/dt = −c·v|v|: one unknown, one column.

The text below, appended to `agents/unit-08.py`, makes a submission that switches the speed loop off from 30 s to 50 s of the race: it still steers, but it commands throttle 0 and brake 0.
(It wraps the agent's own speed loop, `self.speed_pid.update`, so that it returns 0 in the window.)

```python
COAST = '''

COAST_FROM, COAST_TO = 30.0, 50.0  # s: the speed loop is switched off (no throttle, no brake) in between
_Unit08 = RoarCompetitionSolution


class RoarCompetitionSolution(_Unit08):
    async def initialize(self) -> None:
        await super().initialize()
        self.steps = 0
        speed_loop = self.speed_pid.update

        def coasting(error, dt, **kwargs):
            command = speed_loop(error, dt, **kwargs)
            return 0.0 if COAST_FROM <= self.steps * 0.05 < COAST_TO else command

        self.speed_pid.update = coasting

    async def step(self) -> None:
        await super().step()
        self.steps += 1
'''
```

Write that submission to `workdir / "coast.py"` (the text of `agents/unit-08.py` followed by `COAST`, as Unit 1 built its saboteur), and race it on `"test-circuit"`, `laps=1`, `max_seconds=120.0`, `seed=0`, twice: once with `car=carla_like(noise=False)` and once with `car=carla_like(noise=True)`.

For each race, the coast equations are the pairs k → k + 1 where rows k − 1, k and k + 1 are driving rows (`held == 0`) whose logged throttle and brake are both exactly 0 at k − 1 (the lag of Exercise 2) and k; each gives the row (−v̄·|v̄|) and the rate (v[k+1] − v[k]) / 0.05, with v the telemetry's `v`.

Define:

- `coast_races`: a dict from `"quiet"` (noise=False) and `"noisy"` (noise=True) to the `RaceResult`;
- `coast_rows`: a dict from the same keys to the number of coast equations;
- `coast_drag`: a dict from the same keys to the fitted c (1/m);
- `full_fit_drag`: the c of Exercise 2's `full_fit`;
- `compare`: three or four sentences: compare the quiet coast-down's c with the full fit's, and the noisy one's with both. Why can a coast-down on the noisy car be so far off? (The noise is a jitter added to the throttle the car applies, and a throttle cannot go below 0.) What does the full fit have that the coast-down lacks, and what else, besides drag, slows a car that coasts through corners?

**Prediction.** The quiet car should give a drag of the right size; the noisy car's throttle jitter, clipped at 0, is a small push the one-column fit cannot separate from drag.

In [ ]:
COAST = '''

COAST_FROM, COAST_TO = 30.0, 50.0  # s: the speed loop is switched off (no throttle, no brake) in between
_Unit08 = RoarCompetitionSolution


class RoarCompetitionSolution(_Unit08):
    async def initialize(self) -> None:
        await super().initialize()
        self.steps = 0
        speed_loop = self.speed_pid.update

        def coasting(error, dt, **kwargs):
            command = speed_loop(error, dt, **kwargs)
            return 0.0 if COAST_FROM <= self.steps * 0.05 < COAST_TO else command

        self.speed_pid.update = coasting

    async def step(self) -> None:
        await super().step()
        self.steps += 1
'''

coast_path = workdir / "coast.py"
with open(UNIT_08) as source, open(coast_path, "w") as target:
    target.write(source.read() + COAST)


def coast_equations(telemetry):
    # pairs k -> k + 1 with rows k - 1, k, k + 1 driving and throttle = brake = 0 at k - 1 and k
    held, throttle, brake, v = telemetry["held"], telemetry["throttle"], telemetry["brake"], telemetry["v"]
    k = np.arange(1, len(v) - 1)
    driving = (held[k - 1] == 0) & (held[k] == 0) & (held[k + 1] == 0)
    coasting = (throttle[k - 1] == 0.0) & (brake[k - 1] == 0.0) & (throttle[k] == 0.0) & (brake[k] == 0.0)
    k = k[driving & coasting]
    mean = (v[k] + v[k + 1]) / 2.0
    return np.column_stack([-mean * np.abs(mean)]), (v[k + 1] - v[k]) / DT, k


coast_races, coast_rows, coast_drag, coast_speeds = {}, {}, {}, {}
for name, noise in (("quiet", False), ("noisy", True)):
    coast_races[name] = run_race(coast_path, "test-circuit", car=carla_like(noise=noise), laps=1, max_seconds=120.0,
                                 seed=0)
    column, rate, k = coast_equations(coast_races[name].telemetry)
    coast_rows[name] = len(rate)
    coast_drag[name] = float(np.linalg.lstsq(column, rate, rcond=None)[0][0])
    v = coast_races[name].telemetry["v"]
    coast_speeds[name] = (float(v[k[0]]), float(v[k[-1] + 1]))
full_fit_drag = float(full_fit[2])
compare = (
    "The quiet coast-down gives c = 3.93e-4, about 27% above the full fit's 3.08e-4; the noisy one gives a negative "
    "drag, -3.17e-4, as if the air pushed the car, and indeed that car went faster while it coasted (15.8 to "
    "17.7 m/s). "
    "On the noisy car the applied throttle is the command plus a jitter that cannot go below 0, so a commanded 0 is "
    "applied as a small push on average, about the size of the drag itself at 15 m/s (c*v^2 is about 0.1 m/s^2), "
    "and with a single column the fit can only blame it on c. "
    "The full fit has throttle and brake columns that are pushed hard and a wide range of speeds, so drag is a large, "
    "distinct part of thousands of its rows, while the coast-down sees 20 s at 14 to 18 m/s. "
    "And a car that coasts through corners also loses speed to its tyres, which take part of the grip to turn it "
    "(Unit 6): the quiet coast-down blames that on drag too, one reason it reads high."
)

for name in ("quiet", "noisy"):
    result = coast_races[name]
    print(f"{name}: finished {result.finished}, {result.collisions} collisions, {result.total_time:.2f} s; "
          f"{coast_rows[name]} coast equations, speed {coast_speeds[name][0]:.1f} -> {coast_speeds[name][1]:.1f} m/s, "
          f"c = {coast_drag[name]:.3e}")
print(f"the full fit's c = {full_fit_drag:.3e}")
print(compare)

# independent checks: the window holds at most 20 s of coasting rows, and the three numbers' order
for name in ("quiet", "noisy"):
    assert coast_races[name].finished and 300 < coast_rows[name] <= 400
assert full_fit_drag == full_fit[2]
assert 1.1 < coast_drag["quiet"] / full_fit_drag < 1.5
assert coast_drag["noisy"] < 0.0 < full_fit_drag < coast_drag["quiet"]
assert coast_speeds["quiet"][1] < coast_speeds["quiet"][0] and coast_speeds["noisy"][1] > coast_speeds["noisy"][0]

## Exercise 9 - Challenge: a gentler probe

The lesson's probe pushes a section with `GRIP[s] = 3.5`, a planned 12.25 m/s² of cornering.
Would a gentler push do?
Probe sections **0** and **1** of `"test-circuit"` with `GRIP[s] = 2.0` instead (a planned 7.0 m/s²), everything else as in the lesson (`A_LAT = 3.5`, `laps=1`, `max_seconds=300.0`, `seed=0`), and analyse them with Exercise 5's `section_p99`.

Define:

- `gentle`: a dict from the section (0 and 1) to its `RaceResult`;
- `gentle_mu_g`: a dict from the section to the pushed section's p99 (m/s², rounded to 0.01);
- `gentle_kind`: a dict from the section to `"estimate"`, `"lower bound"` or `"no estimate"`, by the lesson's rules;
- `shift`: a dict from the section to `gentle_mu_g[s] - MU_G_TEST_CIRCUIT[s]` (m/s²; the shipped values come from the 3.5 probes);
- `gentle_verdict`: three or four sentences: did both sections still slide? One of the two numbers is close to the 3.5 probe's but the rules call it something else: is that rule too strict, or is it doing its job? When would a gentler probe be the better choice, on a real car?

**Prediction.** 7.0 m/s² is below every section's grip on `test-circuit` but a section can still slide at its tightest corner; where it does not crash the rules give only a lower bound, whatever the number.

In [ ]:
gentle, gentle_mu_g, gentle_kind, shift = {}, {}, {}, {}
for s in (0, 1):
    grip = [1.0] * 6
    grip[s] = 2.0
    gentle[s] = run_race(variant(UNIT_07, f"gentle-{s}", A_LAT=3.5, GRIP=tuple(grip)), "test-circuit",
                         car=carla_like(noise=True), laps=1, max_seconds=300.0, seed=0)
    rows, p99 = section_p99(gentle[s].telemetry, test_circuit)[s]
    gentle_mu_g[s] = round(p99, 2)
    gentle_kind[s] = probe_kind(gentle[s], rows)
    shift[s] = gentle_mu_g[s] - MU_G_TEST_CIRCUIT[s]
gentle_verdict = (
    "No: section 1 still slid (9 collisions) and gives an estimate, 9.76 against the 3.5 probe's 9.72 m/s^2, but "
    "section 0's probe never crashed, so its 7.97 m/s^2 is only a lower bound, though it is within 0.08 of the "
    "3.5 probe's 7.89 (the race was not even clean, 2.71 m off the line at worst, but it never crashed). "
    "The rule is doing its job: nothing in this race shows that section 0 reached the limit, and a lower bound that "
    "happens to sit close to the estimate is luck you could only check with the harder probe. "
    "On a real car, where a crash costs a car and not a respawn, a gentler probe is the better first step: start "
    "gentle and raise the push section by section until the car just slides, and take the first slide as the "
    "estimate."
)

for s in (0, 1):
    print(f"section {s}: {gentle[s].collisions} collisions, finished {gentle[s].finished}, "
          f"max |offset| {max_offset(gentle[s]):.2f} m; mu*g {gentle_mu_g[s]} ({gentle_kind[s]}), "
          f"shift {shift[s]:+.2f} m/s^2")
print(gentle_verdict)

# independent checks: the kinds follow the collisions, and both numbers sit within 0.1 of the 3.5 probes'
assert gentle[0].collisions == 0 and gentle_kind[0] == "lower bound"
assert gentle[1].collisions >= 1 and gentle_kind[1] == "estimate"
assert max(abs(shift[0]), abs(shift[1])) <= 0.1
assert (gentle_mu_g[0], gentle_mu_g[1]) == (7.97, 9.76)